# 🤖 Bitcoin Investment Advisor - LangChain Interactive Demo

This notebook demonstrates the complete LangChain-powered Bitcoin investment advisory system. It allows for interactive conversations, real-time analysis, and automated pipeline execution.

**Features:**
- **Conversational AI Interface**: Chat naturally with the advisor.
- **Real-time Analysis**: Fetches and analyzes the latest Bitcoin news.
- **10-Day Price Forecasting**: Generates price predictions based on market data.
- **Investment Recommendations**: Provides comprehensive, data-driven advice.
- **Conversational Memory**: Remembers context from your previous questions.

## 1. Environment Setup

First, we'll install all the required Python packages listed in `requirements.txt`. This step ensures that our environment has all the necessary libraries to run the agent.

In [ ]:
import sys
import subprocess

print("🔧 Setting up environment...")

# Install dependencies from requirements.txt
try:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], capture_output=True, text=True, check=True)
    print(result.stdout)
    print("\n🎉 All dependencies installed successfully!")
except FileNotFoundError:
    print("❌ Error: 'requirements.txt' not found. Please ensure the file is in the same directory as this notebook.")
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install dependencies. Please check the errors below:\n")
    print(e.stderr)

In [ ]:
import os
import json
import time
from datetime import datetime

# Add the project directory to the Python path to ensure local modules can be imported
if '..' not in sys.path:
    sys.path.append('..')

print("📚 Libraries imported successfully!")
print(f"Python version: {sys.version}")
print(f"Current working directory: {os.getcwd()}")

## 2. Configuration Setup

The agent requires API keys and other settings to function. This next cell will create a `config.json` file.

**🚨 Action Required:** You must edit the generated `config.json` file and replace `"your-gemini-key-here"` with your actual Google AI Studio (Gemini) API key.

In [ ]:
# Load configuration from config.json if it exists, otherwise create a template and ask the user to fill it in.
config_path = 'config.json'
template = {
    "api_keys": {
        "gemini": "your-gemini-key-here"  # <-- IMPORTANT: REPLACE WITH YOUR KEY
    },
    "models": {
        "summarization_model": "gemini-1.5-flash",
        "analysis_model_path": "../main_models/my-awesome-model_final_bitcoin-individual-news-dataset/checkpoint-400",
        "forecast_model_path": "../main_models/qwen_bitcoin_chat_fast_more_longer_explanation_v2/checkpoint-612",
        "advisory_model_path": "../main_models/my-awesome-model_final_bitcoin-investment-advisory-dataset_v2/checkpoint-400"
    },
    "news": {
        "max_articles": 20,
        "short_term_count": 10,
        "long_term_count": 10,
        "sources": [
            "https://feeds.feedburner.com/CoinDesk",
   
            "https://cointelegraph.com/rss",
            "https://bitcoinmagazine.com/feed",
            "https://news.bitcoin.com/feed"
        ]
    },
    "output_dir": "outputs/notebook_demo_results",
    "langchain": {
        "memory_window": 10,
        "max_iterations": 10,
        "timeout": 300,
        "streaming": True
    }
}

config = None
if os.path.exists(config_path):
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
        print(f"✅ Loaded configuration from '{config_path}'.")
    except Exception as e:
        print(f"❌ Failed to read '{config_path}': {e}")
        print('\nCreating a fresh template to help you fix the file...')
        with open(config_path, 'w') as f:
            json.dump(template, f, indent=2)
        config = template
        print(f"📝 Wrote template to '{config_path}'. Please edit it and add your Gemini API key.")
else:
    # Create a template config.json so users can fill in their API keys and settings
    with open(config_path, 'w') as f:
        json.dump(template, f, indent=2)
    config = template
    print(f"📝 No '{config_path}' found. A template has been written to '{config_path}'. Please edit it and add your Gemini API key before continuing.")

# Ensure the output directory exists
os.makedirs(config.get('output_dir', 'outputs/notebook_demo_results'), exist_ok=True)

# Show a short summary to the user
print('\nConfiguration summary:')
print(f"  • Output directory: {config.get('output_dir')}")
print(f"  • News sources count: {len(config.get('news', {}).get('sources', []))}")

# Simple check to warn if the user hasn't set the Gemini key
gemini_key = config.get('api_keys', {}).get('gemini')
if not gemini_key or 'your-gemini-key-here' in str(gemini_key):
    print('\n⚠️  Gemini API key is missing or still the placeholder. Please add your API key to config.json before running the agent.')
else:
    print('\n✅ Gemini API key appears to be configured.')

## 3. Initialize the LangChain Bitcoin Advisor

Now we'll load the `BitcoinLangChainOrchestrator` and initialize it with our configuration. If successful, it will print a list of the specialized tools the agent has at its disposal.

In [ ]:
try:
    from langchain_bitcoin_advisor import BitcoinLangChainOrchestrator
    print("✅ LangChain Bitcoin Advisor imported successfully!")
except ImportError as e:
    print(f"❌ Failed to import BitcoinLangChainOrchestrator: {e}")
    print("Please ensure 'langchain_bitcoin_advisor.py' and 'multi_agent_pipeline.py' are in the same directory.")
    # Stop execution if import fails
    raise e

print("\n🚀 Initializing Bitcoin Investment Advisor...")
try:
    advisor = BitcoinLangChainOrchestrator(config_path=config_path)
    print("\n✅ Advisor initialized successfully!")
    print("\n🤖 Available Tools:")
    for tool in advisor.tools:
        print(f"  - {tool.name}: {tool.description}")
except Exception as e:
    print(f"\n❌ Failed to initialize advisor: {e}")
    print("\n💡 Troubleshooting Tips:")
    print("  1. Did you add your Gemini API key to 'config.json'?")
    print("  2. Are all dependencies installed correctly?")
    # Stop execution if initialization fails
    raise e

## 4. Interactive Conversation Demo

Let's chat with the advisor. We'll create a helper function to send questions and display responses. Then, we'll ask the agent to perform its full analysis workflow step-by-step.

In [ ]:
def chat_with_advisor(question: str):
    """Sends a question to the advisor and prints the streaming response."""
    print(f"\n👤 You: {question}")
    print("\n🤖 Advisor: ", end="", flush=True)
    
    start_time = time.time()
    try:
        # The .invoke() method runs the agent and returns the final response.
        # Because streaming is enabled, the agent's thoughts and tool usage will be printed in real-time.
        response = advisor.agent_executor.invoke({"input": question})
        elapsed_time = time.time() - start_time
        print(f"\n\n⏱️  Response generated in {elapsed_time:.2f} seconds.")
        return response
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
        return None

In [ ]:
print("🧪 Kicking off the full analysis workflow...")
chat_with_advisor("Please run the complete Bitcoin investment advisory pipeline for today. Start by collecting news, then analyze it, create a forecast, and finally give me a detailed investment recommendation.")

## 5. Automated Pipeline Execution

The advisor can also run its entire workflow non-interactively. The `run_automated_pipeline()` method executes all steps in sequence and saves the detailed results to a JSON file.

In [ ]:
print("🚀 Running complete automated Bitcoin analysis pipeline...")

target_date = datetime.now().strftime('%Y-%m-%d')
print(f"📅 Target date: {target_date}")

try:
    results = advisor.run_automated_pipeline(target_date=target_date)
    
    if results['status'] == 'completed':
        print("\n✅ Pipeline completed successfully!")
        
        results_file = os.path.join(config['output_dir'], f"bitcoin_langchain_{target_date}.json")
        print(f"  • Detailed results saved to: {results_file}")
        
        # Show a preview of the final response
        response_preview = results.get('response', '')
        print("\n💬 Response Preview:")
        print(response_preview[:1000] + "..." if len(response_preview) > 1000 else response_preview)
    else:
        print(f"❌ Pipeline failed: {results.get('error', 'Unknown error')}")
except Exception as e:
    print(f"❌ Automated pipeline execution failed: {e}")

## 6. Memory and Context Demonstration

The advisor remembers the conversation. Let's ask a follow-up question that relies on the context of our previous analysis.

In [ ]:
print("🧠 Testing conversational memory...")
chat_with_advisor("Based on the analysis you just performed, what is the single biggest risk factor I should be watching?")

In [ ]:
# You can inspect the conversation history directly
print("📚 Displaying conversation history from memory:")

try:
    history = advisor._get_conversation_history()
    if not history:
        print("No history recorded yet.")
    else:
        for turn in history:
            role = turn['role'].capitalize()
            content_preview = turn['content'][:150].replace('\n', ' ') + '...'
            print(f"- {role}: "{content_preview}"")
except Exception as e:
    print(f"Could not retrieve conversation history: {e}")

## 🎉 Demo Complete!

You have successfully run the LangChain-powered Bitcoin Investment Advisor. Here's a summary of what you accomplished:

1.  **Environment Setup**: Installed all necessary dependencies.
2.  **Configuration**: Created a `config.json` file for your API key.
3.  **Advisor Initialization**: Loaded the conversational agent and its tools.
4.  **Interactive Conversation**: Ran the full analysis pipeline by chatting with the agent.
5.  **Automated Pipeline**: Executed the entire workflow with a single command.
6.  **Memory & Context**: Demonstrated the agent's ability to handle follow-up questions.

### 🚀 Next Steps

- **Ask Your Own Questions**: Modify the `chat_with_advisor` cells to ask your own Bitcoin-related questions.
- **Customize the Configuration**: Edit `config.json` to change news sources or model settings.
- **Explore the Code**: Dive into `langchain_bitcoin_advisor.py` and `multi_agent_pipeline.py` to see how the agents and tools work.